# DATA 715 Lab: Build a Literary Corpus with Spark

**Goal:** Select a book from [Project Gutenberg Australia](https://gutenberg.net.au/) or [Project Gutenberg](https://www.gutenberg.org/ebooks/), retrieve its plain text, and create a reusable text corpus using Hadoop-style MapReduce and PySpark.

**Learning outcomes:** Explain document vs token vs frequency corpus; perform map, filter, reduce; contrast a Python MapReduce simulation with distributed RDDs and DataFrames; examine data quality and lineage; export analysis-ready data.

**Suggested example URL:** `https://gutenberg.net.au/ebooks06/0602581.txt`. Students may select another direct `.txt` link. Check the rights statement for the specific work and your location before redistribution. Gutenberg Australia's public-domain determination is for **Australia**, not necessarily other countries.

**Workflow:** Download → inspect → remove available boilerplate → normalize → tokenize → MapReduce word counts → DataFrame analysis → save corpus.

## 1. Environment setup (Google Colab)
This lab installs PySpark with pip. Java 17 is used where the Colab runtime does not already offer an appropriate JVM. Restart the runtime if you change Spark versions mid-session.

In [ ]:
import os, subprocess, sys
subprocess.run(["apt-get", "update", "-qq"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "openjdk-17-jdk-headless", "-qq"], check=True, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark>=4.0,<5"], check=True)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").appName("DATA715-LiteraryCorpus").getOrCreate()
sc = spark.sparkContext
print("Spark", spark.version, "| parallelism", sc.defaultParallelism)

## 2. Pick a book
Open the source in a browser and select its plain-text URL. Record the title, author, and source. The default is the example URL. If a download is blocked or unavailable, upload a `.txt` file into the Colab Files pane and set `LOCAL_FILE` instead.

In [ ]:
BOOK_URL = "https://gutenberg.net.au/ebooks06/0602581.txt"
BOOK_TITLE = "Selected Gutenberg book"  # Replace after inspecting the text
BOOK_AUTHOR = "Unknown"              # Replace after inspecting the text
BOOK_ID = "book_01"
LOCAL_FILE = None  # e.g. "/content/my_book.txt" to use an uploaded file

from pathlib import Path
import urllib.request
if LOCAL_FILE:
    raw_bytes = Path(LOCAL_FILE).read_bytes()
else:
    req = urllib.request.Request(BOOK_URL, headers={"User-Agent": "DATA715 educational corpus lab"})
    with urllib.request.urlopen(req, timeout=45) as response:
        raw_bytes = response.read()

# Some older Gutenberg Australia files use Western European encodings.
for encoding in ("utf-8-sig", "cp1252"):
    try:
        raw_text = raw_bytes.decode(encoding)
        break
    except UnicodeDecodeError:
        continue
print("Bytes:", len(raw_bytes), "| Characters:", len(raw_text), "| Encoding:", encoding)
print(raw_text[:1500])

## 3. Inspect and prepare document text
Ebooks contain title pages, publisher notices, tables of contents, or rights statements. These can distort word frequencies. **Inspect the text first** and optionally choose the start and end of the literary body. Positions are character indices; leave them unset to retain the whole file. Document any trimming decision.

In [ ]:
START_AT = None   # e.g. raw_text.find("CHAPTER I") after inspection
END_AT = None     # e.g. raw_text.rfind("THE END") after inspection
body = raw_text[START_AT:END_AT]
assert body.strip(), "Selected text is empty"
print("Body characters:", len(body))
print("Beginning:\n", body[:400])
print("Ending:\n", body[-400:])

## 4. Clean and tokenize
A corpus can be a collection of documents, sentences, or tokens. Here we build both a **document-level corpus** and a **token-level corpus**. Cleaning choices change results: this simple English-focused tokenizer removes punctuation, lowercases, and keeps alphabetic words (including apostrophes).

In [ ]:
import re
from collections import Counter

lines = [line.strip() for line in body.splitlines() if line.strip()]
def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

preview = tokenize(lines[0])
print("Nonempty lines:", len(lines), "| First line tokens:", preview[:20])
assert tokenize("Hello, HELLO! Don't stop.") == ["hello", "hello", "don't", "stop"]

## 5. Hadoop-style MapReduce, locally simulated
This is **not a Hadoop cluster**. It demonstrates the algorithm: **map** each word to `(word, 1)`, **shuffle/group** by key, and **reduce** counts. The Spark sections below distribute the same logic.

In [ ]:
tokens = tokenize(body)
mapper_output = [(word, 1) for word in tokens]
counts_python = Counter()
for word, value in mapper_output:
    counts_python[word] += value
print("Tokens:", len(tokens), "| Unique words:", len(counts_python))
print("Top 15:", counts_python.most_common(15))

## 6. Spark RDD: distributed MapReduce
RDD transformations are **immutable** and lazily evaluated. `flatMap` emits words, `map` emits key-value pairs, and `reduceByKey` combines counts. In Colab, local Spark runs on one machine using multiple local tasks.

In [ ]:
lines_rdd = sc.parallelize(lines, numSlices=4)
words_rdd = lines_rdd.flatMap(tokenize)
pairs_rdd = words_rdd.map(lambda word: (word, 1))
counts_rdd = pairs_rdd.reduceByKey(lambda a, b: a + b)
top_rdd = counts_rdd.takeOrdered(15, key=lambda pair: -pair[1])
print("RDD top 15:", top_rdd)
print("RDD lineage:\n", counts_rdd.toDebugString().decode("utf-8", errors="replace"))
assert dict(counts_rdd.collect()) == dict(counts_python)
print("RDD counts match local MapReduce")

## 7. RDD immutability demonstration
Try assigning an item to an RDD. The expected error is caught so students can continue with the lab. `map` creates a **new** RDD.

In [ ]:
try:
    words_rdd[0] = "changed"
except TypeError as exc:
    print("Expected:", exc)

upper_rdd = words_rdd.map(str.upper)
print("Original:", words_rdd.take(5))
print("Transformed:", upper_rdd.take(5))

## 8. Spark DataFrame: a structured token corpus
We add a line index so students can ask **where** words appear. This is a useful corpus rather than only a count table. The per-line token position begins at zero. For large-scale production parsing, avoid Python-side full-book collection; here a single book is intentionally small.

In [ ]:
from pyspark.sql import types as T
indexed = list(enumerate(lines, start=1))
line_df = spark.createDataFrame(indexed, ["line_number", "text"])
tokens_df = (line_df
    .withColumn("tokens", F.split(F.trim(F.regexp_replace(F.lower(F.col("text")), "[^a-z']+", " ")), r"\s+"))
    .select("line_number", F.posexplode("tokens").alias("token_position", "word"))
    .filter(F.col("word") != "")
    .withColumn("book_id", F.lit(BOOK_ID))
    .select("book_id", "line_number", "token_position", "word"))
tokens_df.show(10, truncate=False)
tokens_df.printSchema()
print("Token rows:", tokens_df.count())

**Note:** The RDD tokenizer uses a regular expression; this DataFrame uses a simpler SQL expression. Apostrophes and unusual Unicode punctuation may be handled differently. The next cell makes the difference visible rather than assuming identical results.

In [ ]:
word_counts_df = (tokens_df.groupBy("word")
    .agg(F.count("*").alias("frequency"))
    .orderBy(F.desc("frequency"), F.asc("word")))
word_counts_df.show(20, truncate=False)
print("RDD total tokens:", words_rdd.count())
print("DataFrame total tokens:", tokens_df.count())
print("Difference:", tokens_df.count() - words_rdd.count())

## 9. Build reusable corpus tables
Export a document metadata table, token corpus, and word-frequency table. Parquet stores a schema and is useful for downstream Spark or SQL analysis. Paths are in the Colab runtime and will be lost when the session is reset unless copied to persistent storage.

In [ ]:
output_dir = "/content/literary_corpus"
metadata_df = spark.createDataFrame([(
    BOOK_ID, BOOK_TITLE, BOOK_AUTHOR,
    LOCAL_FILE or BOOK_URL, len(body), len(lines)
)], ["book_id", "title", "author", "source", "characters", "nonempty_lines"])
metadata_df.write.mode("overwrite").parquet(f"{output_dir}/documents")
tokens_df.write.mode("overwrite").parquet(f"{output_dir}/tokens")
word_counts_df.write.mode("overwrite").parquet(f"{output_dir}/word_counts")
print("Saved Parquet corpus under", output_dir)
spark.read.parquet(f"{output_dir}/documents").show(truncate=False)

## 10. Query the corpus with SQL

In [ ]:
tokens_df.createOrReplaceTempView("corpus_tokens")
spark.sql("""
SELECT word, COUNT(*) AS frequency
FROM corpus_tokens
WHERE LENGTH(word) >= 5
GROUP BY word
ORDER BY frequency DESC, word
LIMIT 15
""").show(truncate=False)

## 11. Extension: context and bigrams
Word counts ignore order. A bigram is two adjacent words. The following uses the RDD tokenizer and counts bigrams **within each source line**, not across line boundaries.

In [ ]:
bigrams_rdd = (lines_rdd
    .map(tokenize)
    .flatMap(lambda w: [(w[i], w[i+1]) for i in range(len(w)-1)])
    .map(lambda pair: (pair, 1))
    .reduceByKey(lambda a, b: a+b))
print("Top bigrams:")
for pair, frequency in bigrams_rdd.takeOrdered(15, key=lambda x: -x[1]):
    print(" ".join(pair), frequency)

## 12. Student investigation and deliverables

1. **Choose your own book.** Record its title, author, direct text URL, and the rights statement you checked. Explain what you removed from the header/footer, if anything.
2. **Build the corpus.** Report number of characters, nonempty lines, tokens, and unique words. Distinguish a document corpus from a token corpus and a frequency table.
3. **MapReduce explanation.** Use a 10-word excerpt to show sample mapper pairs and reduced word counts. Explain why `reduceByKey` is preferable to collecting all pairs on the driver for large data.
4. **Compare APIs.** Show the top 10 words from an RDD and a DataFrame. Explain any differences caused by tokenization, not only execution method.
5. **Stop words.** Remove at least five common English words and report how the top 10 changes. Discuss what information the filter removes.
6. **Interpretation.** Identify three frequent content words and discuss whether frequency alone captures their meaning.
7. **Export.** Show the schema of your saved token-level Parquet corpus and a SQL query on the output.

**Optional challenge:** Process two books and add a second `book_id`, then compare normalized word rates (occurrences per 1,000 tokens) rather than raw counts.

**Reflection questions:** What does `collect()` do? Why is an RDD immutable? How does lineage support recovery? Why does caching help repeated Spark actions? Why are Parquet outputs more suitable than raw text for many analytical workloads?

## 13. Optional stop-word starter (students complete)

Set `STOP_WORDS`, filter the DataFrame with `.filter(~F.col("word").isin(...))`, and compare the frequencies before and after. A stop-word list is a modeling decision, not a universal truth.

In [ ]:
STOP_WORDS = ["the", "and", "of", "to", "a"]
filtered_tokens_df = tokens_df.filter(~F.col("word").isin(STOP_WORDS))
(filtered_tokens_df.groupBy("word")
 .count().orderBy(F.desc("count"), "word")
 .show(15, truncate=False))